In [1]:
import pandas as pd
import numpy as np


In [7]:
# 데이터 로드
df = pd.read_csv("mock_data_2.csv").copy()

# 순서와 크소어 소드 매핑
use_frequency_map = {
    "rare": 1,
    "monthly": 2,
    "weekly": 3,
    "frequent": 4
}

last_use_recency_map = {
    ">30d": 1,
    "7-30d": 2,
    "1-7d": 3,
    "<1d": 4,
}

df["use_frequency_score"] = df["use_frequency"].map(use_frequency_map)
df["last_use_recency_score"] = df["last_use_recency"].map(last_use_recency_map)

df["log_monthly_cost"] = np.log1p(df["monthly_cost"]) # 오른쪽 꼬리 완화
df["monthly_cost_z"] = (df["monthly_cost"] - df["monthly_cost"].mean()) / df["monthly_cost"].std(ddof=0)

# value_gap: "사용가치 - 비용부담" 관점
df["value_gap"] = (
    df["use_frequency_score"]
    + df['last_use_recency_score']
    + df['perceived_necessity']
    - df['cost_burden']
)

# rebuy_satisfaction_gap:
# perceived_necessity 대체 프록시로 사용
if "satisfaction" in df.columns:
    df["rebuy_satisfaction_gap"] = df["monthly_cost"] - df["satisfaction"]
else:
    df["rebuy_satisfaction_gap"] = df["monthly_cost"] - df["perceived_necessity"]

# cost_to_necessity_ratio:
df["cost_to_necessity_ratio"] = df["monthly_cost"] * (df["perceived_necessity"] + 1e-6)

df["cost_burden_x_replacement"] = df["cost_burden"] * df["replacement_available"]
df["necessity_x_recency"] = df["perceived_necessity"] * df["last_use_recency_score"]
df["frequency_x_rebuy"] = df["use_frequency_score"] * df["would_rebuy"]

high_cost_threshold = df["monthly_cost"].quantile(0.75)
df["is_high_cost"] = (df["monthly_cost"] > high_cost_threshold).astype(int)

# 이탈 신호 플래그(규칙 기반)
df["has_churn_signal"] = (
    (df["use_frequency_score"] <= 2)
    & (df["last_use_recency_score"] <= 2)
    & (df["cost_burden"] >= 4)
).astype(int)

# 6) 확인
new_cols = [
    "use_frequency_score",
    "last_use_recency_score",
    "value_gap",
    "rebuy_satisfaction_gap",
    "cost_to_necessity_ratio",
    "log_monthly_cost",
    "monthly_cost_z",
    "cost_burden_x_replacement",
    "necessity_x_recency",
    "frequency_x_rebuy",
    "is_high_cost",
    "has_churn_signal",
]
display(df[new_cols].head())
display(df[new_cols].describe().T)

,use_frequency_score,last_use_recency_score,value_gap,rebuy_satisfaction_gap,cost_to_necessity_ratio,log_monthly_cost,monthly_cost_z,cost_burden_x_replacement,necessity_x_recency,frequency_x_rebuy,is_high_cost,has_churn_signal
0,3,4,9,10431,31302.010434,9.252921,-0.557620,0,12,12,0,0
1,3,4,10,9502,38024.009506,9.159784,-0.660058,0,16,15,0,0
2,2,1,0,25875,25876.025876,10.161110,1.146951,0,1,6,1,1
3,2,1,1,31877,95640.031880,10.369766,1.809705,5,3,2,1,1
4,4,4,11,9904,39632.009908,9.201199,-0.615683,1,16,16,0,0


,count,mean,std,min,25%,50%,75%,max
use_frequency_score,20000.0,2.711050e+00,1.111764,1.000000,2.000000,3.000000,4.000000,4.000000
last_use_recency_score,20000.0,2.628600e+00,1.206497,1.000000,1.000000,3.000000,4.000000,4.000000
value_gap,20000.0,6.690300e+00,3.099269,-2.000000,4.000000,7.000000,9.000000,12.000000
rebuy_satisfaction_gap,20000.0,1.548223e+04,9059.342645,2895.000000,8572.750000,12879.500000,21159.000000,58997.000000
cost_to_necessity_ratio,20000.0,5.230157e+04,36975.951074,2900.002900,24822.756723,41862.013954,71198.266910,257930.051586
log_monthly_cost,20000.0,9.472456e+00,0.611147,7.972811,9.056839,9.463703,9.959962,10.985310
monthly_cost_z,20000.0,2.806644e-17,1.000025,-1.389264,-0.762717,-0.287342,0.626484,4.803357
cost_burden_x_replacement,20000.0,1.014650e+00,1.265866,0.000000,0.000000,1.000000,2.000000,5.000000
necessity_x_recency,20000.0,9.305050e+00,5.855255,1.000000,4.000000,8.000000,15.000000,20.000000
frequency_x_rebuy,20000.0,8.229700e+00,5.457603,1.000000,3.000000,8.000000,12.000000,20.000000


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

In [12]:
# =========================
# 3) Feature groups (중복 제거 버전)
# =========================
target_col = "target"
# 명목형 (One-Hot)
nominal_cols = ["subscription_type", "use_frequency", "last_use_recency"]
# 순서형/정수형 (수치 처리)
ordinal_cols = ["perceived_necessity", "cost_burden", "would_rebuy"]
# 연속형 (스케일링 권장)
continuous_cols = [
    "monthly_cost",
    "log_monthly_cost",
    "value_gap",
    "rebuy_satisfaction_gap",
    "cost_to_necessity_ratio",
    "cost_burden_x_replacement",
    "necessity_x_recency",
    "frequency_x_rebuy",
]
# 이진형 (0/1, 스케일링 불필요)
binary_cols = ["replacement_available", "is_high_cost", "has_churn_signal"]
# 존재 컬럼만 필터
nominal_cols = [c for c in nominal_cols if c in df.columns]
ordinal_cols = [c for c in ordinal_cols if c in df.columns]
continuous_cols = [c for c in continuous_cols if c in df.columns]
binary_cols = [c for c in binary_cols if c in df.columns]
X = df[nominal_cols + ordinal_cols + continuous_cols + binary_cols]
y = df[target_col]
# =========================
# 4) Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# =========================
# 5) Preprocess + Logistic
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_cols),
        ("ord_cont", StandardScaler(), ordinal_cols + continuous_cols),
        ("binary", "passthrough", binary_cols),
    ],
    remainder="drop"
)
log_reg = LogisticRegression(
    random_state=42,
    max_iter=3000,
    class_weight="balanced"
)
clf = Pipeline([
    ("preprocess", preprocessor),
    ("model", log_reg),
])
clf.fit(X_train, y_train)
# =========================
# 6) Evaluate
# =========================
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
prec_0 = precision_score(y_test, y_pred, pos_label=0)
rec_0  = recall_score(y_test, y_pred, pos_label=0)
f1_0   = f1_score(y_test, y_pred, pos_label=0)
prec_1 = precision_score(y_test, y_pred, pos_label=1)
rec_1  = recall_score(y_test, y_pred, pos_label=1)
f1_1   = f1_score(y_test, y_pred, pos_label=1)
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC : {auc:.4f}")
print("-" * 50)
print(f"[target=0] Precision: {prec_0:.4f} | Recall: {rec_0:.4f} | F1: {f1_0:.4f}")
print(f"[target=1] Precision: {prec_1:.4f} | Recall: {rec_1:.4f} | F1: {f1_1:.4f}")
print("-" * 50)
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("-" * 50)
print(classification_report(y_test, y_pred, digits=4))

Accuracy: 0.6783
ROC-AUC : 0.6753
--------------------------------------------------
[target=0] Precision: 0.4688 | Recall: 0.5450 | F1: 0.5040
[target=1] Precision: 0.7904 | Recall: 0.7354 | F1: 0.7619
--------------------------------------------------
Confusion Matrix
[[ 654  546]
 [ 741 2059]]
--------------------------------------------------
              precision    recall  f1-score   support

           0     0.4688    0.5450    0.5040      1200
           1     0.7904    0.7354    0.7619      2800

    accuracy                         0.6783      4000
   macro avg     0.6296    0.6402    0.6330      4000
weighted avg     0.6939    0.6783    0.6845      4000



In [14]:
from sklearn.ensemble import RandomForestClassifier

# =========================
# 5) Preprocess + RandomForest
# (RF는 스케일링 불필요)
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_cols),
        ("num_bin", "passthrough", ordinal_cols + continuous_cols + binary_cols),
    ],
    remainder="drop"
)
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=8,
    min_samples_leaf=4,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)
clf_rf = Pipeline([
    ("preprocess", preprocessor),
    ("model", rf),
])
clf_rf.fit(X_train, y_train)
# =========================
# 6) Evaluate
# =========================
y_pred = clf_rf.predict(X_test)
y_proba = clf_rf.predict_proba(X_test)[:, 1]
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
prec_0 = precision_score(y_test, y_pred, pos_label=0)
rec_0  = recall_score(y_test, y_pred, pos_label=0)
f1_0   = f1_score(y_test, y_pred, pos_label=0)
prec_1 = precision_score(y_test, y_pred, pos_label=1)
rec_1  = recall_score(y_test, y_pred, pos_label=1)
f1_1   = f1_score(y_test, y_pred, pos_label=1)
print("=== RandomForest Result ===")
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC : {auc:.4f}")
print("-" * 50)
print(f"[target=0] Precision: {prec_0:.4f} | Recall: {rec_0:.4f} | F1: {f1_0:.4f}")
print(f"[target=1] Precision: {prec_1:.4f} | Recall: {rec_1:.4f} | F1: {f1_1:.4f}")
print("-" * 50)
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("-" * 50)
print(classification_report(y_test, y_pred, digits=4))

=== RandomForest Result ===
Accuracy: 0.7695
ROC-AUC : 0.8122
--------------------------------------------------
[target=0] Precision: 0.6273 | Recall: 0.5708 | F1: 0.5977
[target=1] Precision: 0.8229 | Recall: 0.8546 | F1: 0.8385
--------------------------------------------------
Confusion Matrix
[[ 685  515]
 [ 407 2393]]
--------------------------------------------------
              precision    recall  f1-score   support

           0     0.6273    0.5708    0.5977      1200
           1     0.8229    0.8546    0.8385      2800

    accuracy                         0.7695      4000
   macro avg     0.7251    0.7127    0.7181      4000
weighted avg     0.7642    0.7695    0.7662      4000



In [ ]:
# =========================
# 3) Feature set
# =========================
target_col = "target"
# CatBoost에 직접 범주형으로 넣을 컬럼
cat_cols = ["subscription_type", "use_frequency", "last_use_recency"]
# 나머지 수치형
num_cols = [
    "monthly_cost",
    "perceived_necessity",
    "cost_burden",
    "would_rebuy",
    "replacement_available",
    "log_monthly_cost",
    "value_gap",
    "rebuy_satisfaction_gap",
    "cost_to_necessity_ratio",
    "cost_burden_x_replacement",
    "necessity_x_recency",
    "frequency_x_rebuy",
    "is_high_cost"
    "has_churn_signal",
]
cat_cols = [c for c in cat_cols if c in df.columns]
num_cols = [c for c in num_cols if c in df.columns]
X = df[cat_cols + num_cols].copy()
y = df[target_col].astype(int)
# CatBoost는 categorical dtype/string 처리 안정적
for c in cat_cols:
    X[c] = X[c].astype(str)
# =========================
# 4) Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
cat_features_idx = [X_train.columns.get_loc(c) for c in cat_cols]
# 클래스 불균형(0:1 = 3:7) 보정
neg, pos = np.bincount(y_train)  # neg=0 개수, pos=1 개수
class_weights = [1.0, neg / pos]  # 1 클래스 가중치 상승
# =========================
# 5) Train CatBoost
# =========================
cat_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=1200,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=42,
    class_weights=class_weights,
    verbose=200
)
cat_model.fit(
    X_train, y_train,
    cat_features=cat_features_idx,
    eval_set=(X_test, y_test),
    use_best_model=True
)
# =========================
# 6) Evaluate
# =========================
y_pred = cat_model.predict(X_test).astype(int).ravel()
y_proba = cat_model.predict_proba(X_test)[:, 1]
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
prec_0 = precision_score(y_test, y_pred, pos_label=0)
rec_0  = recall_score(y_test, y_pred, pos_label=0)
f1_0   = f1_score(y_test, y_pred, pos_label=0)
prec_1 = precision_score(y_test, y_pred, pos_label=1)
rec_1  = recall_score(y_test, y_pred, pos_label=1)
f1_1   = f1_score(y_test, y_pred, pos_label=1)
print("=== CatBoost Result ===")
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC : {auc:.4f}")
print("-" * 50)
print(f"[target=0] Precision: {prec_0:.4f} | Recall: {rec_0:.4f} | F1: {f1_0:.4f}")
print(f"[target=1] Precision: {prec_1:.4f} | Recall: {rec_1:.4f} | F1: {f1_1:.4f}")
print("-" * 50)
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("-" * 50)
print(classification_report(y_test, y_pred, digits=4))

0:	test: 0.7298376	best: 0.7298376 (0)	total: 98.9ms	remaining: 1m 58s
200:	test: 0.8084713	best: 0.8084713 (200)	total: 3.23s	remaining: 16.1s
400:	test: 0.8133728	best: 0.8134213 (397)	total: 4.83s	remaining: 9.63s
600:	test: 0.8155604	best: 0.8155604 (600)	total: 6.36s	remaining: 6.34s
800:	test: 0.8175393	best: 0.8175393 (800)	total: 8.1s	remaining: 4.04s
1000:	test: 0.8174216	best: 0.8178007 (901)	total: 9.8s	remaining: 1.95s
1199:	test: 0.8182100	best: 0.8183978 (1123)	total: 11.4s	remaining: 0us

bestTest = 0.8183977679
bestIteration = 1123

Shrink model to first 1124 iterations.
=== CatBoost Result ===
Accuracy: 0.7350
ROC-AUC : 0.8184
--------------------------------------------------
[target=0] Precision: 0.5467 | Recall: 0.6833 | F1: 0.6074
[target=1] Precision: 0.8480 | Recall: 0.7571 | F1: 0.8000
--------------------------------------------------
Confusion Matrix
[[ 820  380]
 [ 680 2120]]
--------------------------------------------------
              precision    recall